Building an End-to-End Data Engineering Pipeline for E-Commerce Order Analytics

Step 1: EXTRACT - Baca Data Mentah

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np  
import ast
# Import semua data raw 
data = pd.read_csv("../data/raw/raw_customers.csv")

data.head(20)

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
0,CUST-0001,customer1@email.com,Farid Saputra,8.334233e+10,2023-10-04 01:01:00,Surabaya,P,premium
1,CUST-0002,customer2@email.com,Jefri Saputra,8.831408e+08,2023-07-08,solo,Laki-laki,Regular
2,CUST-0003,customer3@email.com,Omar Firmansyah,8.244566e+10,2023-01-22 03:22:00,semarang,Laki-laki,Regular
3,CUST-0004,customer4@email.com,YOGA FADILLAH,8.205764e+08,"Sep 14, 2023",YOGYAKARTA,M,Premium
4,CUST-0005,customer5@email.com,Hendra Wijaya,8.207095e+08,04/04/2023,malang,Perempuan,New
5,CUST-0006,customer6@email.com,Rafi Wibowo,8.378329e+10,2023-10-13,Denpasar,P,regular
6,CUST-0007,customer7@email.com,Reza Rahman,8.847525e+08,"Mar 28, 2023",JAKARTA,NaN,Regular
7,CUST-0008,customer8@email.com,Kevin Nugroho,8.235153e+10,2023-11-06 12:56:00,Solo,Laki-laki,Premium
8,CUST-0009,customer9@email.com,Bagas Fadillah,8.675034e+08,2023-09-30 11:14:00,bandung,Perempuan,Regular
9,CUST-0010,customer10@email.com,fajar pratama,8.821761e+10,07/10/2023,malang,M,regular


In [25]:
# Inspeksi awal
print(f"Jumlah baris: {len(data)}")
print(f"Kolom: {list(data.columns)}")
data.info()

Jumlah baris: 55
Kolom: ['customer_id', 'customer_email', 'customer_name', 'phone_number', 'join_date', 'kota', 'gender', 'segment']
<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     54 non-null     str    
 1   customer_email  55 non-null     str    
 2   customer_name   54 non-null     str    
 3   phone_number    50 non-null     float64
 4   join_date       55 non-null     str    
 5   kota            55 non-null     str    
 6   gender          44 non-null     str    
 7   segment         54 non-null     str    
dtypes: float64(1), str(7)
memory usage: 3.6 KB


In [26]:
print("\nDuplikasi")
print(f"{data.duplicated().sum()}")


Duplikasi
0


In [27]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
customer_id        1
customer_email     0
customer_name      1
phone_number       5
join_date          0
kota               0
gender            11
segment            1
dtype: int64


In [28]:
data.nunique()

customer_id       51
customer_email    52
customer_name     50
phone_number      47
join_date         51
kota              25
gender             6
segment            6
dtype: int64

In [29]:
# Nilai Negatif
phone_negatif = data[data['phone_number']< 0]
print(phone_negatif)

Empty DataFrame
Columns: [customer_id, customer_email, customer_name, phone_number, join_date, kota, gender, segment]
Index: []


Step 2: TRANSFORM - Bersihkan Data

In [30]:
# Menentukan kolom yang akan diperiksa
kolom = [
    'customer_id',
    'customer_email',
    'customer_name',
    'phone_number',
    'join_date',
    'kota',
    'gender',
    'segment'
]

# Menampilkan semua baris yang memiliki minimal satu missing value
missing_data = data[data[kolom].isna().any(axis=1)]

display(missing_data)

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
6,CUST-0007,customer7@email.com,Reza Rahman,8.847525e+08,"Mar 28, 2023",JAKARTA,NaN,Regular
10,CUST-0011,customer11@email.com,Eko Rahman,8.115410e+08,19/11/2023,DENPASAR,NaN,New
11,CUST-0012,customer12@email.com,Miftah Pratama,8.838597e+10,2023-08-19,Denpasar,NaN,New
14,CUST-0015,customer15@email.com,Bayu Kusuma,6.282350e+11,2023-02-09 17:10:00,makassar,NaN,premium
17,CUST-0018,customer18@email.com,Andi Kusuma,NaN,09/01/2023,Semarang,M,Premium
22,CUST-0023,customer23@email.com,Arif Hidayat,8.320972e+08,18/06/2023,malang,NaN,regular
24,CUST-0025,customer25@email.com,Fikar Wijaya,NaN,2023-10-01 16:58:00,Denpasar,L,premium
25,CUST-0026,customer26@email.com,JOKO WIBOWO,NaN,2023-04-05 03:56:00,Yogyakarta,M,premium
31,CUST-0032,customer32@email.com,Yudi Rahman,8.121818e+10,2023-10-26 08:02:00,jakarta,NaN,Premium
34,CUST-0035,customer35@email.com,Nizar Hidayat,8.574699e+08,2023-09-18 19:47:00,Bandung,NaN,Premium


In [31]:
# Langsung menhhapus data yang NaN pada customers_id 
data = data.dropna(subset=['customer_id'])

In [32]:
kolom = ['customer_name', 'kota', 'segment']

for col in kolom:
    data[col] = data[col].str.strip().str.title()

display(data[kolom].head(10))

,customer_name,kota,segment
0,Farid Saputra,Surabaya,Premium
1,Jefri Saputra,Solo,Regular
2,Omar Firmansyah,Semarang,Regular
3,Yoga Fadillah,Yogyakarta,Premium
4,Hendra Wijaya,Malang,New
5,Rafi Wibowo,Denpasar,Regular
6,Reza Rahman,Jakarta,Regular
7,Kevin Nugroho,Solo,Premium
8,Bagas Fadillah,Bandung,Regular
9,Fajar Pratama,Malang,Regular


In [33]:
# Menentukan kolom yang akan diperiksa
kolom = [
    'customer_id',
    'customer_email',
    'customer_name',
    'phone_number',
    'join_date',
    'kota',
    'gender',
    'segment'
]

# Menampilkan semua baris yang memiliki minimal satu missing value
missing_data = data[data[kolom].isna().any(axis=1)]

display(missing_data)

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
6,CUST-0007,customer7@email.com,Reza Rahman,8.847525e+08,"Mar 28, 2023",Jakarta,NaN,Regular
10,CUST-0011,customer11@email.com,Eko Rahman,8.115410e+08,19/11/2023,Denpasar,NaN,New
11,CUST-0012,customer12@email.com,Miftah Pratama,8.838597e+10,2023-08-19,Denpasar,NaN,New
14,CUST-0015,customer15@email.com,Bayu Kusuma,6.282350e+11,2023-02-09 17:10:00,Makassar,NaN,Premium
17,CUST-0018,customer18@email.com,Andi Kusuma,NaN,09/01/2023,Semarang,M,Premium
22,CUST-0023,customer23@email.com,Arif Hidayat,8.320972e+08,18/06/2023,Malang,NaN,Regular
24,CUST-0025,customer25@email.com,Fikar Wijaya,NaN,2023-10-01 16:58:00,Denpasar,L,Premium
25,CUST-0026,customer26@email.com,Joko Wibowo,NaN,2023-04-05 03:56:00,Yogyakarta,M,Premium
31,CUST-0032,customer32@email.com,Yudi Rahman,8.121818e+10,2023-10-26 08:02:00,Jakarta,NaN,Premium
34,CUST-0035,customer35@email.com,Nizar Hidayat,8.574699e+08,2023-09-18 19:47:00,Bandung,NaN,Premium


In [34]:
# Mengisi missing value pada kolom customer_email dengan dummy123@email.com 
data['customer_email'] = data['customer_email'].fillna('dummy123@email.com')

In [35]:
# Mengganti missing value customer_name menjadi dummy dengan nama yang  berbeda setiap dokumen
import random

first_names = [
    "Budi", "Andi", "Siti", "Rina", "Dimas",
    "Fajar", "Nadia", "Putri", "Agus", "Indah"
]

last_names = [
    "Santoso", "Pratama", "Saputra", "Wijaya", "Hidayat",
    "Lestari", "Permata", "Kusuma", "Nugroho", "Rahman"
]

mask = data['customer_name'].isna()

data.loc[mask, 'customer_name'] = [
    f"{random.choice(first_names)} {random.choice(last_names)}"
    for _ in range(mask.sum())
]

display(data[['customer_name']])

,customer_name
0,Farid Saputra
1,Jefri Saputra
2,Omar Firmansyah
3,Yoga Fadillah
4,Hendra Wijaya
5,Rafi Wibowo
6,Reza Rahman
7,Kevin Nugroho
8,Bagas Fadillah
9,Fajar Pratama


In [36]:
# Mengisi phone number pada kolom phone_number dan mengganti formatnya dengan standar +
# Mengisi missing value dengan nomor dummy unik
# Mengubah kolom phone_number menjadi string
data['phone_number'] = data['phone_number'].astype('string')
mask = data['phone_number'].isna()
data.loc[mask, 'phone_number'] = [
    f'081200000{i:03d}'
    for i in range(1, mask.sum() + 1)
]
display(data[['phone_number']])

,phone_number
0,83342331444.0
1,883140807.0
2,82445662585.0
3,820576383.0
4,820709497.0
5,83783290795.0
6,884752529.0
7,82351531223.0
8,867503414.0
9,88217611860.0


In [37]:
# Standarisasi format phone number +62
import re
def format_phone(phone):
    if pd.isna(phone):
        return phone
    # Hilangkan karakter selain angka
    phone = re.sub(r'\D', '', str(phone))
    # Mengatasi format scientific notation
    if 'e+' in str(phone).lower():
        phone = str(int(float(phone)))
    if phone.startswith('62'):
        phone = '+' + phone
    elif phone.startswith('0'):
        phone = '+62' + phone[1:]
    elif phone.startswith('8'):
        phone = '+62' + phone
    return phone
data['phone_number'] = data['phone_number'].apply(format_phone)
display(data[['phone_number']])

,phone_number
0,+62833423314440
1,+628831408070
2,+62824456625850
3,+628205763830
4,+628207094970
5,+62837832907950
6,+628847525290
7,+62823515312230
8,+628675034140
9,+62882176118600


In [38]:
# Mengubah format tanggal menjadi standar ISO 2024-06-27
# Mengubah berbagai format tanggal menjadi datetime
data['join_date'] = pd.to_datetime(
    data['join_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)
data['join_date'] = data['join_date'].dt.strftime('%Y-%m-%d')
print(data['join_date'].isna().sum())
data.head()

0


,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
0,CUST-0001,customer1@email.com,Farid Saputra,+62833423314440,2023-04-10,Surabaya,P,Premium
1,CUST-0002,customer2@email.com,Jefri Saputra,+628831408070,2023-08-07,Solo,Laki-laki,Regular
2,CUST-0003,customer3@email.com,Omar Firmansyah,+62824456625850,2023-01-22,Semarang,Laki-laki,Regular
3,CUST-0004,customer4@email.com,Yoga Fadillah,+628205763830,2023-09-14,Yogyakarta,M,Premium
4,CUST-0005,customer5@email.com,Hendra Wijaya,+628207094970,2023-04-04,Malang,Perempuan,New


In [39]:
# Menentukan kolom yang akan diperiksa
kolom = [
    'customer_id',
    'customer_email',
    'customer_name',
    'phone_number',
    'join_date',
    'kota',
    'gender',
    'segment'
]

# Menampilkan semua baris yang memiliki minimal satu missing value
missing_data = data[data[kolom].isna().any(axis=1)]

display(missing_data)

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
6,CUST-0007,customer7@email.com,Reza Rahman,+628847525290,2023-03-28,Jakarta,NaN,Regular
10,CUST-0011,customer11@email.com,Eko Rahman,+628115409560,2023-11-19,Denpasar,NaN,New
11,CUST-0012,customer12@email.com,Miftah Pratama,+62883859722350,2023-08-19,Denpasar,NaN,New
14,CUST-0015,customer15@email.com,Bayu Kusuma,+6282350343240,2023-09-02,Makassar,NaN,Premium
22,CUST-0023,customer23@email.com,Arif Hidayat,+628320972200,2023-06-18,Malang,NaN,Regular
31,CUST-0032,customer32@email.com,Yudi Rahman,+62812181795990,2023-10-26,Jakarta,NaN,Premium
34,CUST-0035,customer35@email.com,Nizar Hidayat,+628574699420,2023-09-18,Bandung,NaN,Premium
35,CUST-0036,customer36@email.com,Siti Rahman,+628545851780,2023-04-07,Makassar,NaN,New
37,CUST-0038,customer38@email.com,Tono Nugroho,+628474635220,2023-06-03,Malang,NaN,Premium
47,CUST-0048,customer48@email.com,Bagas Pratama,+628996007660,2023-09-03,Yogyakarta,NaN,Regular


In [40]:
# Missing value pada kolom gender akan di isi secra random Laki-Laki dan Perempuan 
import random
# Mengisi missing value pada kolom gender secara acak
mask = data['gender'].isna()
data.loc[mask, 'gender'] = [
    random.choice(['Laki-Laki', 'Perempuan'])
    for _ in range(mask.sum())]
# Menampilkan hasil
display(data[['customer_name', 'gender']].head(10))

,customer_name,gender
0,Farid Saputra,P
1,Jefri Saputra,Laki-laki
2,Omar Firmansyah,Laki-laki
3,Yoga Fadillah,M
4,Hendra Wijaya,Perempuan
5,Rafi Wibowo,P
6,Reza Rahman,Laki-Laki
7,Kevin Nugroho,Laki-laki
8,Bagas Fadillah,Perempuan
9,Fajar Pratama,M


In [41]:
# Penulisan gender semua data Normalisasi menjadi Laki-Laki dan Perempuan 
# Menghapus spasi dan mengubah menjadi huruf kecil
data['gender'] = (data['gender'].astype(str).str.strip().str.lower())
# Mapping ke format standar
mapping_gender = {
    'l': 'Laki-Laki',
    'lk': 'Laki-Laki',
    'laki': 'Laki-Laki',
    'laki-laki': 'Laki-Laki',
    'laki laki': 'Laki-Laki',
    'pria': 'Laki-Laki',
    'male': 'Laki-Laki',
    'm': 'Laki-Laki',
    'p': 'Perempuan',
    'pr': 'Perempuan',
    'perempuan': 'Perempuan',
    'wanita': 'Perempuan',
    'female': 'Perempuan',
    'f': 'Perempuan'}
# Mengganti sesuai mapping
data['gender'] = data['gender'].replace(mapping_gender)
# Menampilkan hasil
display(data[['gender']].head(10))

,gender
0,Perempuan
1,Laki-Laki
2,Laki-Laki
3,Laki-Laki
4,Perempuan
5,Perempuan
6,Laki-Laki
7,Laki-Laki
8,Perempuan
9,Laki-Laki


In [42]:
# Menentukan kolom yang akan diperiksa
kolom = [
    'customer_id',
    'customer_email',
    'customer_name',
    'phone_number',
    'join_date',
    'kota',
    'gender',
    'segment'
]
# Menampilkan semua baris yang memiliki minimal satu missing value
missing_data = data[data[kolom].isna().any(axis=1)]

display(missing_data)

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment


In [43]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     54 non-null     str  
 1   customer_email  54 non-null     str  
 2   customer_name   54 non-null     str  
 3   phone_number    54 non-null     str  
 4   join_date       54 non-null     str  
 5   kota            54 non-null     str  
 6   gender          54 non-null     str  
 7   segment         54 non-null     str  
dtypes: str(8)
memory usage: 3.5 KB


In [44]:
data.head(10)

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
0,CUST-0001,customer1@email.com,Farid Saputra,+62833423314440,2023-04-10,Surabaya,Perempuan,Premium
1,CUST-0002,customer2@email.com,Jefri Saputra,+628831408070,2023-08-07,Solo,Laki-Laki,Regular
2,CUST-0003,customer3@email.com,Omar Firmansyah,+62824456625850,2023-01-22,Semarang,Laki-Laki,Regular
3,CUST-0004,customer4@email.com,Yoga Fadillah,+628205763830,2023-09-14,Yogyakarta,Laki-Laki,Premium
4,CUST-0005,customer5@email.com,Hendra Wijaya,+628207094970,2023-04-04,Malang,Perempuan,New
5,CUST-0006,customer6@email.com,Rafi Wibowo,+62837832907950,2023-10-13,Denpasar,Perempuan,Regular
6,CUST-0007,customer7@email.com,Reza Rahman,+628847525290,2023-03-28,Jakarta,Laki-Laki,Regular
7,CUST-0008,customer8@email.com,Kevin Nugroho,+62823515312230,2023-06-11,Solo,Laki-Laki,Premium
8,CUST-0009,customer9@email.com,Bagas Fadillah,+628675034140,2023-09-30,Bandung,Perempuan,Regular
9,CUST-0010,customer10@email.com,Fajar Pratama,+62882176118600,2023-10-07,Malang,Laki-Laki,Regular


In [45]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
customer_id       0
customer_email    0
customer_name     0
phone_number      0
join_date         0
kota              0
gender            0
segment           0
dtype: int64


In [46]:
# Simpan ke CSV data setelah proses atau clean
data.to_csv(
    "../data/warehouse/customers_clean.csv",
    index=False,
    encoding="utf-8"
)
# Melihat kembali data yang disimpan
data = pd.read_csv("../data/warehouse/customers_clean.csv")
data.head()

,customer_id,customer_email,customer_name,phone_number,join_date,kota,gender,segment
0,CUST-0001,customer1@email.com,Farid Saputra,62833423314440,2023-04-10,Surabaya,Perempuan,Premium
1,CUST-0002,customer2@email.com,Jefri Saputra,628831408070,2023-08-07,Solo,Laki-Laki,Regular
2,CUST-0003,customer3@email.com,Omar Firmansyah,62824456625850,2023-01-22,Semarang,Laki-Laki,Regular
3,CUST-0004,customer4@email.com,Yoga Fadillah,628205763830,2023-09-14,Yogyakarta,Laki-Laki,Premium
4,CUST-0005,customer5@email.com,Hendra Wijaya,628207094970,2023-04-04,Malang,Perempuan,New
